KEGG富集分析绘图

In [ ]:
# ============================================================
# 富集分析可视化：弦图 (Chord Diagram) + 点图 (Dot Plot)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PatchCollection
from matplotlib.colors import TwoSlopeNorm
import matplotlib.cm as cm

# ────────────────────────────────────────────────────────────
# 1. 读取数据
# ────────────────────────────────────────────────────────────
df = pd.read_csv(
    './analysis_lu/gigtransformer-rownorm/t2ds_norm_enrichment_pathway.txt',
    sep='\t'
)

# 将 Genes 列从字符串拆分为列表
df['Genes'] = df['Genes'].str.split(', ')

print(f"通路数量: {len(df)}")
print(df[['Term', 'Count', 'PValue', 'FDR']].to_string(index=False))


In [ ]:
# ────────────────────────────────────────────────────────────
# 2. 弦图 (Chord Diagram)
#    展示通路之间共享基因的关系
# ────────────────────────────────────────────────────────────
# 构建通路-基因共享矩阵
terms = df['Term'].tolist()
n = len(terms)

# 计算每对通路的共享基因数
shared = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            set_i = set(df.iloc[i]['Genes'])
            set_j = set(df.iloc[j]['Genes'])
            shared[i, j] = len(set_i & set_j)

# 简短标签（避免标签过长）
short_labels = [t.replace(' signaling pathway', '').replace(' pathway', '')
                for t in terms]

# ── 弦图绘制函数 ──
def chord_diagram(matrix, labels, colors=None, ax=None, gap=0.03, pad=1.5):
    """
    绘制弦图。
    matrix : n×n 对称矩阵，matrix[i,j] 表示 i->j 的权重
    labels : 长度 n 的标签列表
    """
    n = len(labels)
    if colors is None:
        cmap_local = cm.get_cmap('tab10', n)
        colors = [cmap_local(i) for i in range(n)]
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.set_aspect('equal')

    totals = matrix.sum(axis=1) + matrix.sum(axis=0)
    grand_total = totals.sum() / 2  # 每条边计了两次

    # 为每个节点分配圆弧角度范围
    # 圆弧大小正比于该节点参与的总共享量
    two_pi = 2 * np.pi
    arc_angles = []           # [(start, end), ...]
    sub_angles = []           # 每个节点内按目标节点细分的起止角
    row_totals = matrix.sum(axis=1)

    total_weight = row_totals.sum()
    if total_weight == 0:
        total_weight = 1  # 避免除零

    cursor = 0.0
    for i in range(n):
        frac = row_totals[i] / total_weight if total_weight > 0 else 1.0 / n
        arc_size = frac * (two_pi - n * gap)
        arc_angles.append((cursor, cursor + arc_size))
        cursor += arc_size + gap

    # 细分每个节点弧内的子区段（按目标节点分配）
    sub_angles = []
    for i in range(n):
        start, end = arc_angles[i]
        size = end - start
        row_sum = matrix[i].sum()
        subs = {}
        cur = start
        for j in range(n):
            if row_sum > 0:
                s = (matrix[i, j] / row_sum) * size
            else:
                s = 0
            subs[j] = (cur, cur + s)
            cur += s
        sub_angles.append(subs)

    R_outer = 1.0
    R_inner = 0.85

    # 画弧（节点）
    for i in range(n):
        s, e = arc_angles[i]
        theta = np.linspace(s, e, 200)
        x_out = R_outer * np.cos(theta)
        y_out = R_outer * np.sin(theta)
        x_in  = R_inner * np.cos(theta[::-1])
        y_in  = R_inner * np.sin(theta[::-1])
        xs = np.concatenate([x_out, x_in])
        ys = np.concatenate([y_out, y_in])
        ax.fill(xs, ys, color=colors[i], alpha=0.85, zorder=2)

        # 标签
        mid = (s + e) / 2
        label_r = R_outer + 0.12
        ha = 'left' if np.cos(mid) >= 0 else 'right'
        rot = np.degrees(mid)
        if np.cos(mid) < 0:
            rot += 180
        ax.text(label_r * np.cos(mid), label_r * np.sin(mid),
                short_labels[i], fontsize=9,
                ha=ha, va='center', rotation=rot, rotation_mode='anchor')

    # 画弦（bezier 曲线）
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] == 0:
                continue
            s1, e1 = sub_angles[i][j]
            s2, e2 = sub_angles[j][i]
            mid1 = (s1 + e1) / 2
            mid2 = (s2 + e2) / 2

            # 贝塞尔曲线：两端点 + 控制点在圆心
            p1 = np.array([R_inner * np.cos(mid1), R_inner * np.sin(mid1)])
            p2 = np.array([R_inner * np.cos(mid2), R_inner * np.sin(mid2)])
            ctrl = np.array([0.0, 0.0])

            t_vals = np.linspace(0, 1, 200)
            bx = (1 - t_vals)**2 * p1[0] + 2*(1-t_vals)*t_vals*ctrl[0] + t_vals**2*p2[0]
            by = (1 - t_vals)**2 * p1[1] + 2*(1-t_vals)*t_vals*ctrl[1] + t_vals**2*p2[1]

            # 颜色混合
            c1, c2 = colors[i], colors[j]
            blend = tuple((np.array(c1[:3]) + np.array(c2[:3])) / 2)
            weight = matrix[i, j] / matrix.max() if matrix.max() > 0 else 0.5
            lw = 1 + weight * 8

            ax.plot(bx, by, color=blend, alpha=0.5, linewidth=lw, zorder=1)

    ax.set_xlim(-pad, pad)
    ax.set_ylim(-pad, pad)
    ax.axis('off')
    return ax


# ── 绘图 ──
cmap_tab = cm.get_cmap('tab10', n)
colors = [cmap_tab(i) for i in range(n)]

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_aspect('equal')
chord_diagram(shared, short_labels, colors=colors, ax=ax)

# 图例：显示共享基因数
legend_patches = [mpatches.Patch(color=colors[i], label=terms[i]) for i in range(n)]
ax.legend(handles=legend_patches, loc='lower center',
          bbox_to_anchor=(0.5, -0.05), fontsize=9, frameon=False, ncol=1)

ax.set_title('Pathway Enrichment — Chord Diagram\n(Chord width ∝ shared gene count)',
             fontsize=13, pad=20)

plt.tight_layout()
plt.savefig(r'D:\LLFS-GiG-old\image_storage\enrichment_chord_diagram.png',
            dpi=300, bbox_inches='tight')
plt.show()

# 打印共享矩阵供参考
shared_df = pd.DataFrame(shared, index=short_labels, columns=short_labels)
print("\n共享基因数矩阵:")
print(shared_df.to_string())


In [ ]:
import seaborn as sns

df_kegg = pd.read_csv(
    './analysis_lu/gigtransformer-rownorm/t2ds_norm_enrichment_pathway.txt',
    sep='\t'
)
df_kegg = df_kegg.set_index('Term')

# 与 Sankey 图一致的顺序（从上到下），点图 y=0 在底部故倒序
sankey_order = [
    'Cytosolic DNA-sensing pathway',
    'NOD-like receptor signaling pathway',
    'RIG-I-like receptor signaling pathway',
    'Apoptosis',
    'NF-kappa B signaling pathway',
]
# 点图 y 轴从下到上，所以底部 = 列表末尾
plot_order = sankey_order[::-1]   # 底→顶: NF-kappa B … Cytosolic

p_value_list      = [df_kegg.loc[t, 'PValue'] for t in plot_order]
fdr_list          = [df_kegg.loc[t, 'FDR']    for t in plot_order]
ylabels           = plot_order

xlabels = ['P value', 'FDR']
yn, xn  = len(ylabels), len(xlabels)

ylabels_num = list(range(yn)) + list(range(yn))
xlabels_num = yn * [0] + yn * [1]
c = np.array(p_value_list + fdr_list)

fig, ax = plt.subplots(figsize=(10, max(4, yn * 0.9)))
fig.subplots_adjust(left=0.45, right=0.82, top=0.80, bottom=0.05)

ax.set_xlim(-0.5, xn - 0.5)
ax.set_ylim(-0.5, yn - 0.5)
ax.set(xticks=np.arange(xn), yticks=np.arange(yn),
       xticklabels=xlabels, yticklabels=ylabels)

ax.set_xticks(np.arange(xn) - 0.5, minor=True)
ax.set_yticks(np.arange(yn) - 0.5, minor=True)
ax.grid(which='minor', color='#dddddd', linewidth=0.5)
ax.set_aspect('equal', adjustable='box')

R = [0.3] * len(c)
circles = [plt.Circle((xlabels_num[i], ylabels_num[i]), radius=r)
           for i, r in enumerate(R)]

norm = TwoSlopeNorm(vmin=0, vmax=0.05, vcenter=0.025)
col = PatchCollection(circles, array=c, cmap='Oranges_r', norm=norm)
ax.add_collection(col)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(rotation=0, fontsize=11)
plt.yticks(fontsize=9)

sns.despine(ax=ax, left=False, bottom=True, top=False, right=True)

cbar = fig.colorbar(col, ax=ax, shrink=0.6, aspect=15, pad=0.04)
cbar.set_label('P value / FDR', fontsize=9)

ax.set_title('KEGG Pathway Enrichment Dot Plot', fontsize=12, pad=40)

plt.savefig(r'D:\LLFS-GiG-old\image_storage\enrichment_dotplot.png',
            dpi=300, bbox_inches='tight')
plt.show()






In [ ]:
# ────────────────────────────────────────────────────────────
# 4. Sankey 图
#    左侧：通路节点，右侧：基因节点
#    流量宽度：均等（每条通路-基因连接宽度相同）
# ────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from collections import Counter

df_sankey = pd.read_csv(
    './analysis_lu/gigtransformer-rownorm/t2ds_norm_enrichment_pathway.txt',
    sep='\t'
)
df_sankey['Genes'] = df_sankey['Genes'].str.split(', ')

# ── 只保留指定的 5 条通路 ──
SELECTED_TERMS = [
    'NOD-like receptor signaling pathway',
    'RIG-I-like receptor signaling pathway',
    'NF-kappa B signaling pathway',
    'Cytosolic DNA-sensing pathway',
    'Apoptosis',
]
df_sankey = df_sankey[df_sankey['Term'].isin(SELECTED_TERMS)].copy()
# 按指定顺序排列
df_sankey['Term'] = pd.Categorical(df_sankey['Term'], categories=SELECTED_TERMS, ordered=True)
df_sankey = df_sankey.sort_values('Term')

terms = df_sankey['Term'].tolist()

all_genes = [g for genes in df_sankey['Genes'] for g in genes]
gene_counts = Counter(all_genes)

min_pathways = 2
shared_genes = {g for g, cnt in gene_counts.items() if cnt >= min_pathways}
print(f"出现在 ≥{min_pathways} 条通路的基因数: {len(shared_genes)}")

all_nodes = terms + sorted(shared_genes)
node_idx  = {name: i for i, name in enumerate(all_nodes)}
n_terms   = len(terms)

sources, targets, values = [], [], []
for _, row in df_sankey.iterrows():
    term = row['Term']
    for gene in row['Genes']:
        if gene in shared_genes:
            sources.append(node_idx[term])
            targets.append(node_idx[gene])
            values.append(1)

tab_colors = [
    'rgba(31,119,180,0.8)',
    'rgba(255,127,14,0.8)',
    'rgba(44,160,44,0.8)',
    'rgba(214,39,40,0.8)',
    'rgba(148,103,189,0.8)',
]
gene_color = 'rgba(180,180,180,0.5)'

node_colors = tab_colors[:n_terms] + [gene_color] * len(shared_genes)

link_colors = []
for src in sources:
    base = tab_colors[src % len(tab_colors)]
    link_colors.append(base.replace('0.8', '0.3'))

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=all_nodes,
        color=node_colors,
        pad=15,
        thickness=20,
        line=dict(color='white', width=0.5),
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
    )
))

fig.update_layout(
    title_text='Pathway Enrichment — Sankey Diagram<br>'
               '<sup>Flow: pathway → shared gene (appearing in ≥2 pathways)</sup>',
    title_font_size=27,
    font_size=20,
    width=950,
    height=600,
    paper_bgcolor='white',
)

fig.write_image(r'D:\LLFS-GiG-old\image_storage\enrichment_sankey.png',
                scale=12.5)
fig.show()




GO富集分析图

In [ ]:
# ============================================================
# GO 富集分析可视化：Dot Plot + Chord Diagram + Sankey 图
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.collections import PatchCollection
from matplotlib.colors import TwoSlopeNorm
from collections import Counter
import seaborn as sns
import plotly.graph_objects as go

# ────────────────────────────────────────────────────────────
# 1. 读取数据
# ────────────────────────────────────────────────────────────
df = pd.read_csv(
    './analysis_lu/gigtransformer-rownorm/t2ds_norm_enrichment_go_pathway.txt',
    sep='\t'
)
df['Genes'] = df['Genes'].str.split(', ')

# 去掉 Term 中的 GO ID 前缀（如有），仅保留描述文字
df['Term_short'] = df['Term'].str.replace(r'~GO:\d+', '', regex=True).str.strip()

print(f"GO term 数量: {len(df)}")
print(df[['Term_short', 'Count', 'PValue', 'FDR']].to_string(index=False))


In [ ]:
# ────────────────────────────────────────────────────────────
# 2. Dot Plot
# ────────────────────────────────────────────────────────────
df_go = pd.read_csv(
    './analysis_lu/gigtransformer-rownorm/t2ds_norm_enrichment_go_pathway.txt',
    sep='\t'
)
df_go['Term_short'] = df_go['Term'].str.replace(r'~GO:\d+', '', regex=True).str.strip()
df_go = df_go.set_index('Term_short')

# 与 GO Sankey 图一致的顺序（从上到下），点图 y=0 在底部故倒序
sankey_order_go = [
    'pyroptotic inflammatory response',
    'positive regulation of interleukin-1 beta production',
    'apoptotic process',
    'innate immune response',
    'positive regulation of canonical NF-kappaB signal transduction',
]
plot_order_go = sankey_order_go[::-1]   # 底→顶

p_value_list      = [df_go.loc[t, 'PValue'] for t in plot_order_go]
fdr_list          = [df_go.loc[t, 'FDR']    for t in plot_order_go]
ylabels           = plot_order_go

xlabels = ['P value', 'FDR']
yn, xn  = len(ylabels), len(xlabels)

ylabels_num = list(range(yn)) + list(range(yn))
xlabels_num = yn * [0] + yn * [1]
c = np.array(p_value_list + fdr_list)

fig, ax = plt.subplots(figsize=(10, max(4, yn * 0.9)))
fig.subplots_adjust(left=0.45, right=0.82, top=0.80, bottom=0.05)

ax.set_xlim(-0.5, xn - 0.5)
ax.set_ylim(-0.5, yn - 0.5)
ax.set(xticks=np.arange(xn), yticks=np.arange(yn),
       xticklabels=xlabels, yticklabels=ylabels)
ax.set_xticks(np.arange(xn) - 0.5, minor=True)
ax.set_yticks(np.arange(yn) - 0.5, minor=True)
ax.grid(which='minor', color='#dddddd', linewidth=0.5)
ax.set_aspect('equal', adjustable='box')

R = [0.3] * len(c)
circles = [plt.Circle((xlabels_num[i], ylabels_num[i]), radius=R[i])
           for i in range(len(R))]

norm = TwoSlopeNorm(vmin=0, vmax=0.05, vcenter=0.025)
col  = PatchCollection(circles, array=c, cmap='Oranges_r', norm=norm)
ax.add_collection(col)

ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(rotation=0, fontsize=11)
plt.yticks(fontsize=9)
sns.despine(ax=ax, left=False, bottom=True, top=False, right=True)

cbar = fig.colorbar(col, ax=ax, shrink=0.5, aspect=15, pad=0.04)
cbar.set_label('P value / FDR', fontsize=9)

ax.set_title('GO Enrichment Dot Plot\n(color = significance)', fontsize=11, pad=40)

plt.savefig(r'D:\LLFS-GiG-old\image_storage\go_enrichment_dotplot.png',
            dpi=300, bbox_inches='tight')
plt.show()




In [ ]:
# ────────────────────────────────────────────────────────────
# 3. Chord Diagram
# ────────────────────────────────────────────────────────────
terms = df['Term_short'].tolist()
n     = len(terms)

shared = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            shared[i, j] = len(set(df.iloc[i]['Genes']) & set(df.iloc[j]['Genes']))

# 截短标签
short_labels = [t[:40] + '…' if len(t) > 40 else t for t in terms]

def chord_diagram(matrix, labels, colors=None, ax=None, gap=0.03, pad=1.6):
    n = len(labels)
    if colors is None:
        colors = [cm.get_cmap('tab10')(i) for i in range(n)]
    two_pi = 2 * np.pi
    row_totals = matrix.sum(axis=1)
    total_weight = row_totals.sum() or 1

    cursor = 0.0
    arc_angles = []
    for i in range(n):
        frac = row_totals[i] / total_weight
        size = frac * (two_pi - n * gap)
        arc_angles.append((cursor, cursor + size))
        cursor += size + gap

    sub_angles = []
    for i in range(n):
        start, end = arc_angles[i]
        size = end - start
        row_sum = matrix[i].sum() or 1
        subs, cur = {}, start
        for j in range(n):
            s = (matrix[i, j] / row_sum) * size
            subs[j] = (cur, cur + s)
            cur += s
        sub_angles.append(subs)

    R_outer, R_inner = 1.0, 0.85
    for i in range(n):
        s, e = arc_angles[i]
        theta = np.linspace(s, e, 300)
        xs = np.concatenate([R_outer * np.cos(theta), R_inner * np.cos(theta[::-1])])
        ys = np.concatenate([R_outer * np.sin(theta), R_inner * np.sin(theta[::-1])])
        ax.fill(xs, ys, color=colors[i], alpha=0.85, zorder=2)
        mid = (s + e) / 2
        label_r = R_outer + 0.14
        ha  = 'left' if np.cos(mid) >= 0 else 'right'
        rot = np.degrees(mid) + (180 if np.cos(mid) < 0 else 0)
        ax.text(label_r * np.cos(mid), label_r * np.sin(mid),
                labels[i], fontsize=8, ha=ha, va='center',
                rotation=rot, rotation_mode='anchor')

    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i, j] == 0:
                continue
            mid1 = sum(sub_angles[i][j]) / 2
            mid2 = sum(sub_angles[j][i]) / 2
            p1   = np.array([R_inner * np.cos(mid1), R_inner * np.sin(mid1)])
            p2   = np.array([R_inner * np.cos(mid2), R_inner * np.sin(mid2)])
            t    = np.linspace(0, 1, 300)
            bx   = (1-t)**2 * p1[0] + t**2 * p2[0]
            by   = (1-t)**2 * p1[1] + t**2 * p2[1]
            blend = tuple((np.array(colors[i][:3]) + np.array(colors[j][:3])) / 2)
            lw    = 1 + (matrix[i, j] / matrix.max()) * 8
            ax.plot(bx, by, color=blend, alpha=0.45, linewidth=lw, zorder=1)

    ax.set_xlim(-pad, pad); ax.set_ylim(-pad, pad); ax.axis('off')

cmap_tab = cm.get_cmap('Set2', n)
colors   = [cmap_tab(i) for i in range(n)]

fig, ax = plt.subplots(figsize=(13, 13))
ax.set_aspect('equal')
chord_diagram(shared, short_labels, colors=colors, ax=ax)

legend_patches = [mpatches.Patch(color=colors[i], label=terms[i]) for i in range(n)]
ax.legend(handles=legend_patches, loc='lower center',
          bbox_to_anchor=(0.5, -0.05), fontsize=8, frameon=False, ncol=1)
ax.set_title('GO Enrichment — Chord Diagram\n(Chord width ∝ shared gene count)',
             fontsize=13, pad=20)
plt.tight_layout()
plt.savefig(r'D:\LLFS-GiG-old\image_storage\go_enrichment_chord.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ────────────────────────────────────────────────────────────
# 4. Sankey 图
# ────────────────────────────────────────────────────────────
all_genes   = [g for genes in df['Genes'] for g in genes]
gene_counts = Counter(all_genes)

# GO 通路共享基因多，阈值可设为 1（显示全部）或 2（只看共享）
min_pathways = 2
shared_genes = {g for g, cnt in gene_counts.items() if cnt >= min_pathways}
print(f"出现在 ≥{min_pathways} 条 GO term 的基因数: {len(shared_genes)}")

terms_full = df['Term_short'].tolist()
all_nodes  = terms_full + sorted(shared_genes)
node_idx   = {name: i for i, name in enumerate(all_nodes)}
n_terms    = len(terms_full)

sources, targets, values, link_colors = [], [], [], []
palette = [
    'rgba(31,119,180,0.8)',  'rgba(255,127,14,0.8)',
    'rgba(44,160,44,0.8)',   'rgba(214,39,40,0.8)',
    'rgba(148,103,189,0.8)',
]
for idx, row in df.iterrows():
    term = row['Term_short']
    for gene in row['Genes']:
        if gene in shared_genes:
            sources.append(node_idx[term])
            targets.append(node_idx[gene])
            values.append(1)
            link_colors.append(palette[idx % len(palette)].replace('0.8', '0.25'))

node_colors = palette[:n_terms] + ['rgba(180,180,180,0.5)'] * len(shared_genes)

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=all_nodes,
        color=node_colors,
        pad=12,
        thickness=18,
        line=dict(color='white', width=0.5),
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
    )
))

fig.update_layout(
    title_text='GO Enrichment — Sankey Diagram<br>'
               f'<sup>Flow: GO term → shared gene (appearing in ≥{min_pathways} terms)</sup>',
    title_font_size=27,
    font_size=16,
    width=1000,
    height=650,
    paper_bgcolor='white',
)

fig.write_image(r'D:\LLFS-GiG-old\image_storage\go_enrichment_sankey.png', scale=2)
fig.show()


In [ ]:
# ────────────────────────────────────────────────────────────
# 4. Sankey 图
# ────────────────────────────────────────────────────────────
all_genes   = [g for genes in df['Genes'] for g in genes]
gene_counts = Counter(all_genes)

min_pathways = 2
shared_genes = {g for g, cnt in gene_counts.items() if cnt >= min_pathways}
print(f"出现在 ≥{min_pathways} 条 GO term 的基因数: {len(shared_genes)}")

terms_full   = df['Term_short'].tolist()
genes_sorted = sorted(shared_genes)

# 通路标签用 HTML 放大，基因标签保持原始字符串
TERM_FONTSIZE = 20   # 通路节点字号，按需调整
all_labels = (
    [f'<span style="font-size:{TERM_FONTSIZE}px">{t}</span>' for t in terms_full]
    + genes_sorted
)

# node_idx 仍用原始名称做 key，避免 HTML 标签混入索引
all_nodes_raw = terms_full + genes_sorted
node_idx  = {name: i for i, name in enumerate(all_nodes_raw)}
n_terms   = len(terms_full)

sources, targets, values, link_colors = [], [], [], []
palette = [
    'rgba(31,119,180,0.8)',  'rgba(255,127,14,0.8)',
    'rgba(44,160,44,0.8)',   'rgba(214,39,40,0.8)',
    'rgba(148,103,189,0.8)',
]
for idx, row in df.iterrows():
    term = row['Term_short']
    for gene in row['Genes']:
        if gene in shared_genes:
            sources.append(node_idx[term])
            targets.append(node_idx[gene])
            values.append(1)
            link_colors.append(palette[idx % len(palette)].replace('0.8', '0.25'))

node_colors = palette[:n_terms] + ['rgba(180,180,180,0.5)'] * len(shared_genes)

fig = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        label=all_labels,       # HTML 标签，通路字号更大
        color=node_colors,
        pad=10,
        thickness=16,
        line=dict(color='white', width=0.5),
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
    )
))

fig.update_layout(
    title_text='GO Enrichment — Sankey Diagram<br>'
               f'<sup>Flow: GO term → shared gene (appearing in ≥{min_pathways} terms)</sup>',
    title_font_size=27,
    font_size=15,       # 基因节点字号
    width=1000,
    height=650,
    paper_bgcolor='white',
)

fig.write_image(r'D:\LLFS-GiG-old\image_storage\go_enrichment_sankey.png', scale=2)
fig.show()


## 143 GIG 基因富集分析 — 气泡图 + 条形图（KEGG / GO-BP / GO-MF / GO-CC）